# 🏋️ Fine-Tuning de BioMistral-7B com MedQuAD

**Tech Challenge FIAP - Fase 3 | Assistente Médico Inteligente**

⚠️ **VERSÃO SEGURA**: Este notebook salva TUDO no Google Drive desde o início.
Se a sessão cair, basta reabrir e retomar com `resume_from_checkpoint=True`.

---

## 🎯 O que este notebook faz

Pega o modelo **BioMistral-7B** (já pré-treinado em PubMed) e ajusta ele especificamente com perguntas/respostas médicas do dataset **MedQuAD anonimizado**.

## ⏱️ Tempo estimado

- Setup: ~5 min
- Download modelo: ~10 min
- Treinamento: ~2-4h (2 epochs, A100)
- Salvar no Drive: ~1 min
- **TOTAL: ~3-5h** (pode deixar rodando sem supervisão após os primeiros 10 min)

## 🛡️ Segurança

- ✅ Tudo salvo em `/content/drive/MyDrive/techchallenge_fase3/`
- ✅ Checkpoints a cada epoch no Drive
- ✅ Retomável se cair (via `resume_from_checkpoint`)

---

# SEÇÃO 1: Setup do ambiente + Verificação GPU

**Antes de começar**:
1. Runtime → Change runtime type → Hardware: **A100 GPU**
2. Confirme que está rodando esta célula abaixo

In [ ]:
# ============================================================
# SEÇÃO 1: Setup + Verificação GPU
# ============================================================
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print("=" * 60)
print(f"🖥️  GPU: {torch.cuda.get_device_name(0)}")
print(f"🖥️  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"🐍  Python: {torch.__version__}")
print("=" * 60)

if torch.cuda.get_device_properties(0).total_memory / 1e9 < 30:
    print("⚠️  AVISO: GPU tem menos de 30GB. Recomendado A100 (40GB).")
    print("   Se cair em OOM, reduza per_device_train_batch_size pra 1")
else:
    print("✅ GPU perfeita pra BioMistral-7B + QLoRA!")

Mon Aug 31 19:44:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             44W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

**Esperado**:
```
🖥️  GPU: NVIDIA A100-SXM4-40GB
🖥️  VRAM: 40.0 GB
✅ GPU perfeita pra BioMistral-7B + QLoRA!
```

❌ Se aparecer GPU diferente ou erro: volte em **Runtime → Change runtime type**

---

# SEÇÃO 2: Instalar dependências

In [ ]:
# ============================================================
# SEÇÃO 2: Instalar dependências (~2 min)
# ============================================================
# Usa subprocess em vez de !pip pra evitar problemas de parsing no Colab
import subprocess
import sys

print("📦 Instalando dependências... (2-3 min)")
print("   (vai aparecer muito output, é normal)")

# 1. PyTorch (versão mais recente compatível com o ambiente atual)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch",
    "--index-url", "https://download.pytorch.org/whl/cu124"
], check=True)
print("   ✅ torch instalado")

# 2. Pacotes HuggingFace (sem trava de versão para suportar Python 3.13)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers",
    "datasets",
    "peft",
    "bitsandbytes",
    "accelerate",
    "trl",
    "loguru",
], check=True)
print("   ✅ transformers, peft, bitsandbytes instalados")

# 3. Unsloth (instalação via git)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth[colab-new]",
    "git+https://github.com/unslothai/unsloth.git"
], check=True)
print("   ✅ unsloth instalado")

print("\n✅ Todas as dependências instaladas!")

📦 Instalando dependências... (2-3 min)
   (vai aparecer muito output, é normal)
   ✅ torch instalado
   ✅ transformers, peft, bitsandbytes instalados
   ✅ unsloth instalado

✅ Todas as dependências instaladas!


---

# SEÇÃO 3: Montar Google Drive (PERSISTÊNCIA)

⚠️ **MUITO IMPORTANTE**: vamos trabalhar **dentro do Drive** desde o início.
Isso garante que checkpoints e modelo final sobrevivam se a sessão cair.

In [ ]:
# ============================================================
# SEÇÃO 3: Montar Drive + configurar paths persistentes
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Diretório de trabalho no Drive (PERSISTE mesmo se sessão cair)
WORKDIR = '/content/drive/MyDrive/techchallenge_fase3'
import os
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

# Subdiretórios
CKPT_DIR = f"{WORKDIR}/checkpoints"          # checkpoints intermediários
MODEL_DIR = f"{WORKDIR}/biomistral-medquad-lora"  # modelo final
DATA_DIR = f"{WORKDIR}/data"                # onde vai o train.jsonl

for d in [CKPT_DIR, MODEL_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Working directory: {WORKDIR}")
print(f"   Checkpoints:  {CKPT_DIR}")
print(f"   Modelo final: {MODEL_DIR}")
print(f"   Dados:        {DATA_DIR}")
print()
print(f"📂 Conteúdo atual de {WORKDIR}:")
!ls -lh {WORKDIR}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Working directory: /content/drive/MyDrive/techchallenge_fase3
   Checkpoints:  /content/drive/MyDrive/techchallenge_fase3/checkpoints
   Modelo final: /content/drive/MyDrive/techchallenge_fase3/biomistral-medquad-lora
   Dados:        /content/drive/MyDrive/techchallenge_fase3/data

📂 Conteúdo atual de /content/drive/MyDrive/techchallenge_fase3:
total 12K
drwx------ 2 root root 4.0K Aug 31 19:49 biomistral-medquad-lora
drwx------ 2 root root 4.0K Aug 31 19:49 checkpoints
drwx------ 2 root root 4.0K Aug 31 19:49 data


## ⚠️ Agora faça upload do `train.jsonl` para o Drive

**Como fazer** (escolha uma):

### Opção A: Via PC → Drive (recomendado)
1. No seu PC, copie `C:\Users\Teste\Downloads\Techchalleng3\data\processed\train.jsonl` (17 MB)
2. Cole em `G:\Meu Drive\techchallenge_fase3\data\train.jsonl` (no seu Drive local)
3. Drive sincroniza automaticamente

### Opção B: Upload direto no Colab
```python
from google.colab import files
uploaded = files.upload()  # vai pedir pra selecionar arquivo
import shutil
shutil.move(list(uploaded.keys())[0], f"{DATA_DIR}/train.jsonl")
```

Depois de upar, **rode a célula abaixo** pra confirmar:

In [ ]:
# ============================================================
# SEÇÃO 3.1: Verificar que train.jsonl está no Drive
# ============================================================
import os

train_path = f"{DATA_DIR}/train.jsonl"
if os.path.exists(train_path):
    size_mb = os.path.getsize(train_path) / 1024 / 1024
    with open(train_path) as f:
        n_lines = sum(1 for _ in f)
    print(f"✅ train.jsonl encontrado!")
    print(f"   Tamanho: {size_mb:.1f} MB")
    print(f"   Amostras: {n_lines:,}")
else:
    print(f"❌ train.jsonl NÃO encontrado em {train_path}")
    print(f"   Faça upload antes de continuar (veja célula acima)")
    raise FileNotFoundError("Upload train.jsonl primeiro!")

✅ train.jsonl encontrado!
   Tamanho: 17.1 MB
   Amostras: 14,692


---

# SEÇÃO 4: Carregar BioMistral-7B

Aqui é onde a mágica acontece. Vamos baixar o BioMistral-7B (~14GB) e configurar com QLoRA.

**O que é QLoRA**: Quantização do modelo base para 4-bit + adaptadores LoRA treináveis.
Em vez de ajustar os 7 bilhões de parâmetros, ajustamos apenas ~40 milhões (0.6% do total).

⏱️ **Esta célula leva ~5-10min na primeira vez** (download do modelo)

In [ ]:
# ============================================================
# SEÇÃO 4: Carregar BioMistral-7B com QLoRA
# ============================================================
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096  # tokens máximos por exemplo (instruction + input + output)
DTYPE = None           # auto-detectar (float16 ou bfloat16)
LOAD_IN_4BIT = True    # QLoRA: quantizar pra 4-bit (economiza 4x VRAM)

print("📥 Baixando BioMistral-7B... (pode levar 5-10 min na primeira vez)")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="BioMistral/BioMistral-7B",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    # token="hf_XXXXXXXXXX",  # descomente se precisar autenticar no HuggingFace
)

print("\n✅ BioMistral-7B carregado!")
print(f"   Vocabulário: {tokenizer.vocab_size:,} tokens")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
📥 Baixando BioMistral-7B... (pode levar 5-10 min na primeira vez)
==((====))==  Unsloth 2026.8.22: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Error during conversion: ReadTimeout('The read operation timed out')
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/lib/python3.13/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
  File "/usr/local/lib/python3.13/dist-packages/transformers/safetensors_conversion.py", line 77, in get_conversion_pr_reference
    raise OSError(
    ...<2 lines>...
    )
OSError: Could not create safetensors conversion PR. The repo does not appear t

Unsloth: BioMistral/BioMistral-7B has no pad_token. Using pad_token = <unk>.

✅ BioMistral-7B carregado!
   Vocabulário: 32,000 tokens


---

# SEÇÃO 5: Configurar LoRA adapters

Agora vamos adicionar os 'notas adesivas' (LoRA) ao modelo.

In [ ]:
# ============================================================
# SEÇÃO 5: Configurar LoRA adapters
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # rank LoRA (compromisso performance/velocidade)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,  # fator de escala (= 2 * rank)
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = model.num_parameters()
pct = 100 * trainable_params / all_params

print("=" * 60)
print("📊 PARÂMETROS DO MODELO")
print("=" * 60)
print(f"  Total:           {all_params:,} ({all_params/1e9:.2f}B)")
print(f"  Treináveis:      {trainable_params:,} ({trainable_params/1e6:.1f}M)")
print(f"  % Treinável:     {pct:.4f}%")
print()
print("💡 Apenas ~0.6% dos parâmetros são treinados!")
print("   Por isso LoRA é tão rápido e leve.")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.8.22 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


📊 PARÂMETROS DO MODELO
  Total:           7,283,675,136 (7.28B)
  Treináveis:      41,943,040 (41.9M)
  % Treinável:     0.5758%

💡 Apenas ~0.6% dos parâmetros são treinados!
   Por isso LoRA é tão rápido e leve.


---

# SEÇÃO 6: Carregar e formatar dataset

In [ ]:
# ============================================================
# SEÇÃO 6: Carregar dataset + template Alpaca
# ============================================================
from datasets import load_dataset

dataset = load_dataset("json", data_files=f"{DATA_DIR}/train.jsonl", split="train")
print(f"✅ Dataset carregado: {len(dataset):,} amostras")
print()
print(f"📝 Exemplo (amostra 0):")
print(f"   Instruction: {dataset[0]['instruction']}")
print(f"   Input:       {dataset[0]['input']}")
print(f"   Output:      {dataset[0]['output'][:200]}...")

Generating train split: 0 examples [00:00, ? examples/s]

✅ Dataset carregado: 14,692 amostras

📝 Exemplo (amostra 0):
   Instruction: Is Russell-Silver syndrome inherited ?
   Input:       Context / Topic: Russell-Silver syndrome
   Output:      Most cases of Russell-Silver syndrome are sporadic, which means they occur in people with no history of the disorder in their family. Less commonly, Russell-Silver syndrome can run in families. In som...


In [ ]:
# ============================================================
# SEÇÃO 6.1: Aplicar template Alpaca
# ============================================================
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token


def format_example(example):
    """Formata 1 exemplo no template Alpaca."""
    text = alpaca_prompt.format(
        example["instruction"],
        example.get("input", ""),
        example["output"]
    ) + EOS_TOKEN
    return {"text": text}


dataset = dataset.map(format_example)

print("=" * 60)
print("✅ Dataset formatado no template Alpaca")
print("=" * 60)
print()
print(f"📝 Exemplo formatado (amostra 0):\n")
print(dataset[0]["text"][:600])
print("...")

Map:   0%|          | 0/14692 [00:00<?, ? examples/s]

✅ Dataset formatado no template Alpaca

📝 Exemplo formatado (amostra 0):

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Is Russell-Silver syndrome inherited ?

### Input:
Context / Topic: Russell-Silver syndrome

### Response:
Most cases of Russell-Silver syndrome are sporadic, which means they occur in people with no history of the disorder in their family. Less commonly, Russell-Silver syndrome can run in families. In some affected families, the condition appears to have an autosomal dominant pattern of inheritance. Autosomal dominant i
...


---

# SEÇÃO 7: Configurar o Trainer

In [ ]:
# ============================================================
# SEÇÃO 7: Configurar SFTTrainer
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,         # batch efetivo = 8
        warmup_steps=50,
        num_train_epochs=2,                   # 2 epochs é o sweet spot
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=20,                      # log a cada 20 steps
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir=CKPT_DIR,                   # ← SALVA NO DRIVE!
        save_strategy="epoch",                # salva após cada epoch
        save_total_limit=2,                   # mantém 2 últimos
        report_to="none",                     # sem wandb
    ),
)

print("✅ Trainer configurado!")
print(f"   Dataset:      {len(dataset):,} amostras")
print(f"   Batch efetivo: 2 × 4 = 8")
print(f"   Steps/epoch:  {len(dataset)//8:,}")
print(f"   Total steps:  {len(dataset)//8*2:,} (2 epochs)")
print(f"   Checkpoints:  {CKPT_DIR}")
print(f"   Tempo estimado A100: 2-4h")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/14692 [00:00<?, ? examples/s]

✅ Trainer configurado!
   Dataset:      14,692 amostras
   Batch efetivo: 2 × 4 = 8
   Steps/epoch:  1,836
   Total steps:  3,672 (2 epochs)
   Checkpoints:  /content/drive/MyDrive/techchallenge_fase3/checkpoints
   Tempo estimado A100: 2-4h


---

# SEÇÃO 8: 🚀 TREINAR! (essa é a parte demorada)

Agora vamos **rodar o fine-tuning**. A célula abaixo pode levar 2-4h.

**Dicas:**
- ✅ Pode **minimizar** a janela — o treino continua
- ✅ Pode **fechar** o navegador — o treino continua!
- ✅ Verá mensagens de progresso a cada 20 steps
- ✅ Se cair, basta reabrir e usar `resume_from_checkpoint=True`

**⚠️ Importante**: Se você quer **retomar de onde parou** (caso tenha caído antes), mude a última linha para:
```python
trainer.train(resume_from_checkpoint=True)
```

In [ ]:
# ============================================================
# SEÇÃO 8: TREINAMENTO 🚀
# ============================================================
print("=" * 60)
print("🚀 INICIANDO TREINAMENTO")
print("=" * 60)
print("⏱️  Tempo estimado: 2-4 horas em A100")
print(f"💾 Checkpoints salvos em: {CKPT_DIR}")
print("💡 Pode fechar o navegador — o treino continua!")
print()

# ⚠️ Se você REINICIOU após queda, descomente a linha abaixo:
# trainer.train(resume_from_checkpoint=True)

trainer.train()

print()
print("=" * 60)
print("✅ TREINAMENTO CONCLUÍDO!")
print("=" * 60)

🚀 INICIANDO TREINAMENTO
⏱️  Tempo estimado: 2-4 horas em A100
💾 Checkpoints salvos em: /content/drive/MyDrive/techchallenge_fase3/checkpoints
💡 Pode fechar o navegador — o treino continua!



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 14,692 | Num Epochs = 2 | Total steps = 3,674
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
20,1.425015
40,0.910815
60,0.872208
80,0.836784
100,0.802327
120,0.766477
140,0.820915
160,0.790022
180,0.782128
200,0.799095


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/techchallenge_fase3/checkpoints/checkpoint-1837/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/techchallenge_fase3/checkpoints/checkpoint-1837.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/techchallenge_fase3/checkpoints/checkpoint-3674/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/techchallenge_fase3/checkpoints/checkpoint-3674.



✅ TREINAMENTO CONCLUÍDO!


---

# SEÇÃO 9: Salvar modelo final no Drive

Após o treino, salvamos os adaptadores LoRA (são só ~80MB) no Drive.

In [ ]:
# ============================================================
# SEÇÃO 9: Salvar modelo final no Drive
# ============================================================
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("=" * 60)
print("💾 MODELO SALVO!")
print("=" * 60)
print(f"📂 Diretório: {MODEL_DIR}")
print()
!ls -lh {MODEL_DIR}/

import os
size_mb = sum(
    os.path.getsize(os.path.join(MODEL_DIR, f))
    for f in os.listdir(MODEL_DIR)
) / 1024 / 1024

print(f"\n📦 Tamanho total: {size_mb:.1f} MB")
print("   (Bem menor que o modelo completo de 14GB!)")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/techchallenge_fase3/biomistral-medquad-lora/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/techchallenge_fase3/biomistral-medquad-lora.


💾 MODELO SALVO!
📂 Diretório: /content/drive/MyDrive/techchallenge_fase3/biomistral-medquad-lora

total 164M
-rw------- 1 root root 1.3K Aug 31 22:20 adapter_config.json
-rw------- 1 root root 161M Aug 31 22:20 adapter_model.safetensors
-rw------- 1 root root  477 Aug 31 22:20 chat_template.jinja
-rw------- 1 root root 5.2K Aug 31 22:20 README.md
-rw------- 1 root root  965 Aug 31 22:20 tokenizer_config.json
-rw------- 1 root root 3.4M Aug 31 22:20 tokenizer.json
-rw------- 1 root root 482K Aug 31 19:51 tokenizer.model

📦 Tamanho total: 163.9 MB
   (Bem menor que o modelo completo de 14GB!)


---

# SEÇÃO 10: Testar o modelo com perguntas reais

In [ ]:
# ============================================================
# SEÇÃO 10: Inferência — testar o modelo
# ============================================================
from transformers import TextStreamer

# Ativar modo inferência do Unsloth (mais rápido)
FastLanguageModel.for_inference(model)

def perguntar(instruction: str, topic: str = ""):
    """Faz uma pergunta ao modelo fine-tuned."""
    input_text = alpaca_prompt.format(
        instruction,
        f"Context / Topic: {topic}" if topic else "",
        ""
    )

    inputs = tokenizer([input_text], return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    print(f"\n{'='*70}")
    print(f"❓ PERGUNTA: {instruction}")
    if topic:
        print(f"📋 TÓPICO: {topic}")
    print(f"{'='*70}")
    print(f"🤖 RESPOSTA:")

    _ = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.2,
    )
    print()

# Testar com 3 perguntas
perguntar(
    "What are the symptoms of diabetes type 2?",
    "Diabetes Type 2"
)

perguntar(
    "What are the treatments for high blood pressure?",
    "Hypertension"
)

perguntar(
    "Is breast cancer hereditary?",
    "Breast Cancer"
)

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



❓ PERGUNTA: What are the symptoms of diabetes type 2?
📋 TÓPICO: Diabetes Type 2
🤖 RESPOSTA:
People with diabetes have high blood glucose (sugar) levels because their bodies do not make enough insulin or cannot use insulin effectively. Symptoms may include excessive thirst and urination; increased appetite; weight loss; fatigue; blurred vision; slow healing of wounds; sores in mouth; and dry, itchy skin. People who develop complications from diabetes can also experience numbness or tingling sensations in the feet; headaches; heart problems; kidney disease; eye damage; dental disease; hearing loss; leg amputation; depression; and erectile dysfunction.


❓ PERGUNTA: What are the treatments for high blood pressure?
📋 TÓPICO: Hypertension
🤖 RESPOSTA:


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Medications and lifestyle changes can effectively control most cases of hypertension. Medications Angiotensin-Converting Enzyme (ACE) Inhibitors ACE inhibitors block angiotensin II from being produced in the body. This reduces narrowing or tightening of blood vessels and lowers blood pressure. The medications used to treat this condition include lisinopril (Princepleen), ramipril (Haltelite), enalapril (Aggrastat), captopril, and trandolapril (Meridia). All of these drugs also act as diuretics. Captopril acts primarily on the kidneys to reduce fluid levels; however it may cause dry coughs in some people. Benazepril (Lozolam) is another medication that works like other ACE inhibitors but has been shown to be effective even at very low dosages. Diuretics Diuretics help your body get rid of extra sodium (salt) and water. Water stays in the body because it carries needed nutrients to all parts of the body. However, too much water causes higher blood pressure. Taking diuretics makes you uri

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Breast cancer can be caused by inherited gene mutations (changes), but most cases are not inherited. Changes in several genes have been associated with increased risk of breast cancer; however, only high-risk gene changes account for a significant percentage of all breast cancer cases. The rest of the time, people who develop this disease do not have an identified gene change known to increase their chances of developing it.



---

# SEÇÃO 11: Avaliação quantitativa (perplexity no val set)

In [ ]:
# ============================================================
# SEÇÃO 11: Avaliação — perplexity no val set
# ============================================================
import math
from datasets import load_dataset

# Carregar val.jsonl do Drive (deve estar lá se você upou train.jsonl completo)
VAL_FILE = f"{DATA_DIR}/val.jsonl"

if not os.path.exists(VAL_FILE):
    print(f"⚠️  val.jsonl não encontrado em {VAL_FILE}")
    print(f"   Se você só upou train.jsonl, copie val.jsonl também do seu PC.")
    print(f"   Val.jsonl está em: C:\\Users\\Teste\\Downloads\\Techchalleng3\\data\\processed\\val.jsonl")
else:
    # Correção: Quando carregamos um arquivo avulso, o split padrão é "train", mesmo que o arquivo se chame val
    val_dataset = load_dataset("json", data_files=VAL_FILE, split="train")
    print(f"📂 Carregado val set: {len(val_dataset):,} amostras\n")

    total_loss = 0
    n_samples = 0
    MAX_EVAL_SAMPLES = 100

    model.eval()
    with torch.no_grad():
        for i, example in enumerate(val_dataset):
            if i >= MAX_EVAL_SAMPLES:
                break

            text = alpaca_prompt.format(
                example["instruction"],
                example.get("input", ""),
                example["output"]
            )
            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            ).to("cuda")

            outputs = model(**inputs, labels=inputs["input_ids"])
            total_loss += outputs.loss.item()
            n_samples += 1

            if (i + 1) % 20 == 0:
                print(f"   Avaliado {i+1}/{MAX_EVAL_SAMPLES} amostras...")

    perplexity = math.exp(total_loss / n_samples)

    print()
    print("=" * 60)
    print("📊 RESULTADO DA AVALIAÇÃO")
    print("=" * 60)
    print(f"  Loss média:    {total_loss / n_samples:.4f}")
    print(f"  Perplexity:    {perplexity:.2f}")
    print()
    print("💡 Interpretação:")
    print("   • < 5   = Modelo 'decorou' o val set")
    print("   • 5-15 = Excelente (aprendeu o domínio)")
    print("   • 15-30 = Bom")
    print("   • > 50 = Modelo ainda não aprendeu")

📂 Carregado val set: 816 amostras

   Avaliado 20/100 amostras...
   Avaliado 40/100 amostras...
   Avaliado 60/100 amostras...
   Avaliado 80/100 amostras...
   Avaliado 100/100 amostras...

📊 RESULTADO DA AVALIAÇÃO
  Loss média:    0.5864
  Perplexity:    1.80

💡 Interpretação:
   • < 5   = Modelo 'decorou' o val set
   • 5-15 = Excelente (aprendeu o domínio)
   • 15-30 = Bom
   • > 50 = Modelo ainda não aprendeu


---

# SEÇÃO 12: Avaliação qualitativa — gerar respostas para revisão

In [ ]:
# ============================================================
# SEÇÃO 12: Avaliação qualitativa — comparar com gabarito
# ============================================================
import json
import os

TEST_FILE = f"{DATA_DIR}/test.jsonl"
EVAL_OUTPUT = f"{WORKDIR}/eval_results_qualitativo.json"

if not os.path.exists(TEST_FILE):
    print(f"⚠️  test.jsonl não encontrado. Pulando esta seção.")
else:
    # Correção: Assim como no val set, o split padrão para arquivos únicos é "train"
    test_dataset = load_dataset("json", data_files=TEST_FILE, split="train")

    # Pegar 20 amostras aleatórias
    import random
    rng = random.Random(42)
    indices = rng.sample(range(len(test_dataset)), min(20, len(test_dataset)))
    samples = [test_dataset[i] for i in indices]

    results = []
    print(f"🚀 Gerando respostas para {len(samples)} perguntas...\n")

    for i, example in enumerate(samples):
        input_text = alpaca_prompt.format(
            example["instruction"],
            example.get("input", ""),
            ""
        )
        inputs = tokenizer([input_text], return_tensors="pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2,
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = response.split("### Response:")[-1].strip()

        results.append({
            "instruction": example["instruction"],
            "expected": example["output"][:500],
            "generated": response[:500],
        })

        print(f"  [{i+1}/20] {example['instruction'][:60]}... OK")

    # Salvar
    with open(EVAL_OUTPUT, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Resultados salvos em: {EVAL_OUTPUT}")
    print(f"   {len(results)} pares (esperado vs gerado) para revisão manual")

    # Mostrar 2 exemplos lado a lado
    print("\n" + "=" * 70)
    print("📋 EXEMPLOS LADO A LADO")
    print("=" * 70)
    for r in results[:2]:
        print(f"\n--- PERGUNTA ---")
        print(f"   {r['instruction']}")
        print(f"\n--- ESPERADO ---")
        print(f"   {r['expected'][:300]}...")
        print(f"\n--- GERADO ---")
        print(f"   {r['generated'][:300]}...")
        print()


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🚀 Gerando respostas para 20 perguntas...



Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [1/20] What is (are) Platelet Disorders ?... OK


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [17/20] What is (are) Kidney Cysts ?... OK


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [18/20] How many people are affected by Stickler syndrome ?... OK


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [19/20] What is (are) Leukodystrophies ?... OK
  [20/20] What are the symptoms of Familial juvenile hyperuricaemic ne... OK

✅ Resultados salvos em: /content/drive/MyDrive/techchallenge_fase3/eval_results_qualitativo.json
   20 pares (esperado vs gerado) para revisão manual

📋 EXEMPLOS LADO A LADO

--- PERGUNTA ---
   What is (are) Platelet Disorders ?

--- ESPERADO ---
   Platelets are little pieces of blood cells. Platelets help wounds heal and prevent bleeding by forming blood clots. Your bone marrow makes platelets. Problems can result from having too few or too many platelets, or from platelets that do not work properly. If your blood has a low number of platelet...

--- GERADO ---
   Platelets are blood cell fragments that form in bone marrow. They float around in your blood and help stop bleeding by sticking together to seal off small cuts or breaks on blood vessel walls. There are many types of platelet disorders, including - Thrombocytopenia -- not enough platelets in the bl

In [ ]:
# ============================================================
# SEÇÃO 13.1: Teste de generalização (15 perguntas)
# ============================================================
from unsloth import FastLanguageModel
import torch

# Re-ativar modo inferência
FastLanguageModel.for_inference(model)

def perguntar(instruction: str, topic: str = ""):
    """Faz uma pergunta ao modelo fine-tuned."""
    input_text = alpaca_prompt.format(
        instruction,
        f"Context / Topic: {topic}" if topic else "",
        ""
    )
    inputs = tokenizer([input_text], return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=***
        temperature=0.5,        # mais determinístico p/ teste
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.3,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()

# 15 perguntas de teste (3 categorias)
TESTES = [
    # === CATEGORIA 1: Perguntas FORA do MedQuAD (gerais) ===
    ("What are the early signs of lung cancer?", "Lung Cancer"),
    ("How is multiple sclerosis diagnosed?", "Multiple Sclerosis"),
    ("What lifestyle changes help prevent heart disease?", "Heart Disease Prevention"),
    ("What is the difference between Type 1 and Type 2 diabetes?", "Diabetes Comparison"),
    ("What are the warning signs of a stroke?", "Stroke Symptoms"),

    # === CATEGORIA 2: Doenças modernas (NÃO no MedQuAD) ===
    ("What are the main symptoms of COVID-19?", "COVID-19"),
    ("How does the mRNA vaccine work?", "mRNA Vaccines"),
    ("What is the treatment for dengue hemorrhagic fever?", "Dengue"),
    ("What are the symptoms of monkeypox?", "Monkeypox"),
    ("How is Zika virus transmitted?", "Zika Virus"),

    # === CATEGORIA 3: Edge cases (robustez) ===
    ("o que é diabetes?", ""),  # português!
    ("tell me about aspirin", ""),  # sem tópico estruturado
    ("asdfghjkl", ""),  # gibberish
    ("What is the meaning of life?", ""),  # fora do escopo médico
    ("", ""),  # pergunta vazia
]

print("=" * 70)
print("🧪 TESTE DE GENERALIZAÇÃO - 15 PERGUNTAS")
print("=" * 70)
print()

resultados = []
for i, (q, topic) in enumerate(TESTES, 1):
    print(f"\n{'─' * 70}")
    print(f"[{i:2d}/15] Categoria: {'GERAL' if i <= 5 else 'MODERNA' if i <= 10 else 'EDGE'}")
    print(f"❓ {q}")
    print(f"📋 Tópico: {topic or '(vazio)'}")
    print(f"🤖 Resposta:")

    try:
        resp = perguntar(q, topic)
        # Mostrar primeiros 400 chars
        print(f"   {resp[:400]}{'...' if len(resp) > 400 else ''}")
        resultados.append({"q": q, "topic": topic, "resp": resp, "len": len(resp)})
    except Exception as e:
        print(f"   ❌ ERRO: {e}")
        resultados.append({"q": q, "topic": topic, "resp": f"ERRO: {e}", "len": 0})

print("\n" + "=" * 70)
print("📊 ANÁLISE AUTOMÁTICA")
print("=" * 70)

# Análise simples
vazias = sum(1 for r in resultados if r["len"] < 20)
repeticoes = sum(1 for r in resultados if r["q"].lower() in r["resp"].lower()[:200])
longas = sum(1 for r in resultados if r["len"] > 100)
print(f"\nTotal: 15 perguntas")
print(f"✅ Respostas longas (>100 chars): {longas}/15")
print(f"⚠️  Respostas vazias/curtas: {vazias}/15")
print(f"🔁 Repetindo a pergunta: {repeticoes}/15")

print("\n🎯 VEREDITO:")
if vazias > 5:
    print("   ❌ OVERFITTING SEVERO - modelo não sabe responder fora do treino")
elif repeticoes > 3:
    print("   ⚠️  OVERFITTING MODERADO - modelo repete padrões do treino")
elif longas >= 12:
    print("   ✅ GENERALIZOU BEM - responde com confiança em dados novos")
else:
    print("   ⚠️  RESULTADO MISTO - analisar manualmente abaixo")

print("\n" + "=" * 70)
print("💾 SALVANDO RESULTADOS")
print("=" * 70)
import json
EVAL_FILE = f"{WORKDIR}/test_generalizacao.json"
with open(EVAL_FILE, "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)
print(f"✅ Salvos em: {EVAL_FILE}")

SyntaxError: invalid syntax (3777867436.py, line 20)

In [ ]:
# ============================================================
# SEÇÃO 13.2: Comparação modelo FINE-TUNED vs BASE
# ============================================================
import gc
from unsloth import FastLanguageModel
import torch

print("=" * 70)
print("🔬 COMPARAÇÃO: FINE-TUNED vs BASE (mesmas perguntas)")
print("=" * 70)

# Salvar referência ao modelo fine-tuned
print("\n1️⃣ Carregando modelo BASE (BioMistral-7B sem fine-tuning)...")
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="BioMistral/BioMistral-7B",
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)
print("✅ Modelo base carregado")

def perguntar_base(prompt_template, llm, tok, instruction, topic=""):
    input_text = prompt_template.format(
        instruction,
        f"Context / Topic: {topic}" if topic else "",
        ""
    )
    inputs = tok([input_text], return_tensors="pt").to("cuda")
    outputs = llm.generate(
        **inputs,
        max_new_tokens=***
        temperature=0.5,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.3,
    )
    response = tok.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()

PERGUNTAS_TESTE = [
    ("What are the early signs of lung cancer?", "Lung Cancer"),
    ("How does the mRNA vaccine work?", "mRNA Vaccines"),
    ("o que é diabetes?", ""),
]

for i, (q, topic) in enumerate(PERGUNTAS_TESTE, 1):
    print(f"\n{'─' * 70}")
    print(f"[{i}] PERGUNTA: {q}")
    if topic:
        print(f"    Tópico: {topic}")

    print(f"\n    🤖 BASE (sem fine-tuning):")
    try:
        resp_base = perguntar_base(alpaca_prompt, base_model, base_tokenizer, q, topic)
        print(f"       {resp_base[:250]}{'...' if len(resp_base) > 250 else ''}")
    except Exception as e:
        print(f"       ❌ ERRO: {e}")

print("\n" + "=" * 70)
print("💡 ANÁLISE MANUAL:")
print("=" * 70)
print("""
Compare as respostas dos 2 modelos:

✅ SEU MODELO melhor que o base SE:
   - Resposta é mais clínica e estruturada
   - Menos alucinações / divagações
   - Tom similar ao MedQuAD (formal, informativo)

🤷 SE FOREM SIMILARES:
   - O fine-tuning não agregou muito (dataset MedQuAD é genérico)
   - Mas ainda é válido pra Tech Challenge (mostra que não degradou)

❌ SE SEU MODELO for PIOR:
   - Houve overfitting sério
   - Considere retreinar com mais regularization
""")

---

# 🎉 CONCLUSÃO

Parabéns! Você completou o **fine-tuning do BioMistral-7B** com o dataset MedQuAD.

## 📁 O que você tem agora

1. **Modelo fine-tuned**: `biomistral-medquad-lora/` (~80MB) — pronto pra usar no pipeline LangChain
2. **Resultados qualitativos**: `eval_results_qualitativo.json` — 20 pares pra revisar
3. **Checkpoints**: `checkpoints/` — versões intermediárias
4. **Relatórios do treino**: loss/perplexity impressos

## 🚀 Próximos passos

1. **Baixar modelo pro seu PC**:
   - Pasta `biomistral-medquad-lora/` no Drive
   - Copiar pra `C:\\Users\\Teste\\Downloads\\Techchalleng3\\biomistral-medquad-lora\\`
   - Rodar `python src/ui/gradio_app.py` — vai detectar e usar modelo real!

2. **Testar a UI Gradio** (localmente) com modelo real

3. **Deploy em HuggingFace Spaces** (opcional, URL pública)

4. **Gravar vídeo demo** (≤15min) pro Tech Challenge

## 📚 Documentação

- **DOCX do projeto**: `TECHCHALLENGE_FASE3_PROJETO_COMPLETO.docx` no repo
- **Manual UI**: `MANUAL_UI.md` no repo
- **Relatório técnico**: `RELATORIO_TECNICO_PARA_EQUIPE.md` no repo